# Tarea 2: Ruteo Consistente de Vehículos


In [28]:
!pip install gurobipy pandas


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [29]:
# Imports
import math
import random
from pathlib import Path

In [30]:

# Rutas relativas a la carpeta desde la que se ejecuta el notebook.
PATH_SOLOMON = Path("data/originales")
PATH_GENERADAS = Path("data/instancias")

ARCHIVO_R108 = PATH_SOLOMON / "R108.txt"
ARCHIVO_R110 = PATH_SOLOMON / "R110.txt"

# Comprobar que existen los originales.
for archivo in [ARCHIVO_R108, ARCHIVO_R110]:
    if not archivo.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {archivo.resolve()}"
        )

print("Entrada:", PATH_SOLOMON.resolve())
print("Salida:", PATH_GENERADAS.resolve())

Entrada: /home/pixoll/github/T2opti/data/originales
Salida: /home/pixoll/github/T2opti/data/instancias


# Sección 1 Lectura de archivos originales

In [31]:
def leer_solomon(path):
    """Lee cabecera VEHICLE y filas CUSTOMER, sin depender de espacios exactos."""
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    vehicles = capacity = None
    rows = {}
    in_customers = False
    for line in lines:
        s = line.split()
        if not s:
            continue
        if s[0].upper() == "CUSTOMER":
            in_customers = True
            continue
        if not in_customers and len(s) == 2:
            try:
                vehicles, capacity = int(s[0]), float(s[1])
            except ValueError:
                pass
        if in_customers and len(s) == 7:
            try:
                row = tuple(float(x) for x in s)
            except ValueError:
                continue
            i = int(row[0])
            if i != row[0] or i in rows:
                raise ValueError("Identificador no entero o duplicado")
            if row[3] < 0 or row[4] > row[5] or row[6] < 0:
                raise ValueError(f"Datos invalidos en cliente {i}")
            rows[i] = row
    if 0 not in rows or vehicles is None or capacity is None:
        raise ValueError(f"Formato Solomon incompleto: {path}")
    return rows, vehicles, capacity


def distancia(rows, i, j):
    return math.hypot(
        rows[i][1] - rows[j][1],
        rows[i][2] - rows[j][2]
    )

# Sección 2 Generación de instancias propias

In [49]:
def generar(source, destination, n, days, vehicles, seed, slack):
    rows, original_vehicles, original_q = leer_solomon(source)
    ids = sorted(i for i in rows if i != 0)[:n]  # Solo los primeros n clientes, sin el depot (0).
    # Validaciones de parametros 
    if len(ids) != n or n < 15 or days < 3 or not 3 <= vehicles <= n:
        raise ValueError("Se requieren >=15 clientes, >=3 dias y 3<=vehiculos<=clientes")
    if not 0 <= slack <= 1:
        raise ValueError("El margen de capacidad debe estar entre 0 y 1")

    # Generacion de demanda diaria aleatoria, con al menos un dia activo por cliente.    
    rng = random.Random(seed)
    demand = {}
    for i in ids:
        active = set(rng.sample(range(days), rng.choice([days - 1, days])))
        demand[i] = [max(1, round(rows[i][3] * rng.uniform(0.8, 1.2)))
        if t in active else 0 for t in range(days)]

    # Agrupacion geografica sencilla para construir UNA solución factible de referencia.
    ordered = sorted(
        ids, key=lambda i: math.atan2(
            rows[i][2] - rows[0][2],
            rows[i][1] - rows[0][1]
        )
    )

    # Equilibrio de numero de clientes     
    groups = [
        ordered[r * n // vehicles:(r + 1) * n // vehicles]
        for r in range(vehicles)
    ]

    # Calculo de cargas diarias para cada grupo.
    loads = [
        [sum(demand[i][t] for i in group) for t in range(days)]
        for group in groups
    ]
    # Toma la mayor carga y se redondea hacia arriba con el margen de slack para obtener la nueva capacidad Q.
    q = math.ceil(max(max(x) for x in loads) * (1 + slack))

    # Preparar horarios de apertura y cierre, y ventanas de servicio para cada cliente.
    opening, original_close = rows[0][4:6]
    service = {i: max(1.0, rows[i][6]) for i in ids}
    ready = {i: rows[i][4] for i in ids}
    due = {i: rows[i][5] for i in ids}

    schedules = []
    closing = original_close

    # Construir los recorridos diarios 
    for r, group in enumerate(groups, 1):
        for t in range(days):
            # Recorre cada ruta y cada día. Las rutas se numeran desde 1.
            remaining = {i for i in group if demand[i][t] > 0}
            previous, clock, visits = 0, opening, []
            while remaining:
                # Escoge el siguiente cliente que podria empezar antes. 
                # inicio posible = max{apertura del cliente, hora actual + tiempo de viaje}, si hay empate el identificador mas bajo gana.
                i = min(
                    remaining, key=lambda j: (
                        max(ready[j], clock + distancia(rows, previous, j)), j
                    )
                )
                # Calcula el inicio de servicio y actualiza la ventana de cierre si es necesario.
                start = max(ready[i], clock + distancia(rows, previous, i))
                due[i] = max(due[i], math.ceil(start))

                # Registra la visita, suma el servicio, actualiza la ubicación y elimina al cliente de los pendientes.
                visits.append((i, start))
                clock, previous = start + service[i], i
                remaining.remove(i)

            # Volver al deposito, calcula el regreso y amplía el cierre del depósito si hace falta.
            return_time = clock + distancia(rows, previous, 0)
            closing = max(closing, math.ceil(return_time))
            schedules.append((r, t, visits, return_time))

    # La siguiente parte recorre los horarios construidos y los comprueba.
    # Validacion independiente de la solución factible de referencia: cobertura, consistencia, carga y tiempos.
    for t in range(days):
        seen = []
        for r, day, visits, return_time in schedules:
            if day != t:
                continue
            previous, clock, load = 0, opening, 0
            for i, start in visits:
                if i not in groups[r - 1] or demand[i][t] <= 0:
                    raise ValueError("solución factible de referencia inconsistente")
                if not ready[i] <= start <= due[i]:
                    raise ValueError("Ventana incumplida")
                if start + 1e-8 < clock + distancia(rows, previous, i):
                    raise ValueError("Tiempo de viaje incumplido")
                seen.append(i)
                load += demand[i][t]
                previous, clock = i, start + service[i]
            if load > q or return_time > closing:
                raise ValueError("Capacidad u horario incumplido")
            if abs(return_time - clock - distancia(rows, previous, 0)) > 1e-8:
                raise ValueError("Retorno inconsistente")
        expected = sorted(i for i in ids if demand[i][t] > 0)
        if sorted(seen) != expected:
            raise ValueError("Cobertura diaria incorrecta")

    destination.parent.mkdir(parents=True, exist_ok=True)
    name = destination.stem
    text = [
        f"NAME {name}", "FORMAT CONVRP_MULTIDAY_V1",
        f"SOURCE {source.name}",
        f"SEED {seed}", f"DAYS {days}", f"VEHICLES {vehicles}",
        f"CAPACITY {q}", "DISTANCE EUCLIDEAN_DOUBLE",
        "TRAVEL_TIME EQUAL_DISTANCE", "TIME_UNIT ABSTRACT",
        "", "DEPOT", "ID X Y OPEN CLOSE",
        f"0 {rows[0][1]:g} {rows[0][2]:g} {opening:g} {closing:g}",
        "", "CUSTOMERS", "ID X Y READY DUE SERVICE"
    ]
    for i in ids:
        text.append(
            f"{i} {rows[i][1]:g} {rows[i][2]:g} "
            f"{ready[i]:g} {due[i]:g} {service[i]:g}"
        )
    text += ["", "DAILY_DEMAND", "ID " + " ".join(f"DAY{t + 1}" for t in range(days))]
    text += [f"{i} " + " ".join(map(str, demand[i])) for i in ids]
    text += ["", "EOF"]
    destination.write_text("\n".join(text) + "\n", encoding="utf-8")

    widened = [i for i in ids if due[i] != rows[i][5]]
    report = [
        f"Instancia: {name}", "solución factible de referencia VERIFICADO (no es una solucion optima)",
        f"Dificultad computacional: ver results/{name}_configuraciones.csv",
        f"Vehiculos originales: {original_vehicles}; nuevos: {vehicles}",
        f"Capacidad original: {original_q:g}; nueva: {q}; margen: {slack:.1%}",
        f"Cierre original: {original_close:g}; nuevo: {closing:g}",
        f"Ventanas con limite superior ampliado: {len(widened)}", ""
    ]
    for i in widened:
        report.append(f"Cliente {i}: DUE {rows[i][5]:g} -> {due[i]:g}")
    for r, group in enumerate(groups, 1):
        report += [f"\nAsignacion solución factible de referencia ruta {r}: {group}", f"Cargas: {loads[r - 1]}"]
    for r, t, visits, return_time in schedules:
        route = " -> ".join(["0"] + [str(i) for i, _ in visits] + ["0"])
        starts = ", ".join(f"{i}: {w:.6f}" for i, w in visits)
        report += [
            f"\nRuta {r}, dia {t + 1}: {route}",
            f"Salida fija: {opening:g}; retorno: {return_time:.6f}",
            f"Inicio de servicio (valores mostrados redondeados): {starts}"
        ]
    destination.with_name(name + "_verificacion.txt").write_text(
        "\n".join(report) + "\n", encoding="utf-8"
    )
    print(
        f"{destination.name}: {n} clientes, {days} dias, {vehicles} vehiculos, "
        f"Q={q}, cierre={closing:g}, ventanas ampliadas={len(widened)}; solución factible de referencia OK"
    )

In [33]:
ARCHIVO_I1 = PATH_GENERADAS / "I1.txt"
ARCHIVO_I2 = PATH_GENERADAS / "I2.txt"

generar(
    source=ARCHIVO_R108,
    destination=ARCHIVO_I1,
    n=20,
    days=3,
    vehicles=3,
    seed=42,
    slack=0.05,
)

generar(
    source=ARCHIVO_R110,
    destination=ARCHIVO_I2,
    n=30,
    days=3,
    vehicles=3,
    seed=43,
    slack=0.05,
)

I1.txt: 20 clientes, 3 dias, 3 vehiculos, Q=98, cierre=230, ventanas ampliadas=2; solución factible de referencia OK
I2.txt: 30 clientes, 3 dias, 3 vehiculos, Q=154, cierre=296, ventanas ampliadas=13; solución factible de referencia OK


# Sección 3 Lector de instancias propias

In [34]:
def leer_multidia(path):
    """Lee el formato CONVRP_MULTIDAY_V1 generado por nuestro código."""
    path = Path(path)

    cabecera = {}
    clientes = {}
    demandas = {}
    deposito = None
    seccion = None

    for numero_linea, linea in enumerate(
        path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        linea = linea.strip()

        if not linea or linea.startswith("#"):
            continue

        if linea == "EOF":
            break

        if linea in {"DEPOT", "CUSTOMERS", "DAILY_DEMAND"}:
            seccion = linea
            continue

        partes = linea.split()

        # Ignorar títulos de columnas.
        if partes[0] == "ID":
            continue

        try:
            if seccion is None:
                clave, valor = linea.split(maxsplit=1)
                cabecera[clave] = valor

            elif seccion == "DEPOT":
                if len(partes) != 5 or deposito is not None:
                    raise ValueError("Debe existir una única fila de depósito")

                identificador = int(partes[0])
                x, y, apertura, cierre = map(float, partes[1:])

                deposito = {
                    "id": identificador,
                    "x": x,
                    "y": y,
                    "apertura": apertura,
                    "cierre": cierre,
                }

            elif seccion == "CUSTOMERS":
                if len(partes) != 6:
                    raise ValueError("Fila de cliente con columnas incorrectas")

                i = int(partes[0])

                if i in clientes:
                    raise ValueError(f"Cliente duplicado: {i}")

                x, y, inicio, fin, servicio = map(float, partes[1:])

                clientes[i] = {
                    "x": x,
                    "y": y,
                    "inicio": inicio,
                    "fin": fin,
                    "servicio": servicio,
                }

            elif seccion == "DAILY_DEMAND":
                i = int(partes[0])

                if i in demandas:
                    raise ValueError(f"Demanda duplicada para cliente {i}")

                demandas[i] = list(map(float, partes[1:]))

        except ValueError as error:
            raise ValueError(
                f"{path.name}, línea {numero_linea}: {error}"
            ) from error

    # Comprobaciones de estructura.
    if cabecera.get("FORMAT") != "CONVRP_MULTIDAY_V1":
        raise ValueError("Formato multidía no reconocido")

    dias = int(cabecera["DAYS"])
    vehiculos = int(cabecera["VEHICLES"])
    capacidad = float(cabecera["CAPACITY"])

    if dias <= 0 or vehiculos <= 0 or capacidad <= 0:
        raise ValueError("Días, vehículos y capacidad deben ser positivos")

    if deposito is None or deposito["id"] != 0:
        raise ValueError("Falta el depósito con identificador 0")

    if not 0 <= deposito["apertura"] <= deposito["cierre"]:
        raise ValueError("Horario del depósito inválido")

    n = len(clientes)

    # Nuestros archivos generados seleccionan los clientes 1,...,n.
    if n == 0 or set(clientes) != set(range(1, n + 1)):
        raise ValueError("Se esperan clientes numerados desde 1 hasta n")

    if set(demandas) != set(clientes):
        raise ValueError("Los clientes y las filas de demanda no coinciden")

    for i in clientes:
        c = clientes[i]

        if not 0 <= c["inicio"] <= c["fin"]:
            raise ValueError(f"Ventana inválida del cliente {i}")

        if c["servicio"] <= 0:
            raise ValueError(f"Servicio no positivo del cliente {i}")

        if len(demandas[i]) != dias:
            raise ValueError(f"Cantidad de días incorrecta del cliente {i}")

        if any(q < 0 for q in demandas[i]):
            raise ValueError(f"Demanda negativa del cliente {i}")

    if (
        cabecera.get("DISTANCE") != "EUCLIDEAN_DOUBLE"
        or cabecera.get("TRAVEL_TIME") != "EQUAL_DISTANCE"
    ):
        raise ValueError("Convención de distancia o tiempo no compatible")

    # Conjuntos del modelo.
    C = list(range(1, n + 1))
    D = list(range(1, dias + 1))
    R = list(range(1, vehiculos + 1))

    retorno = n + 1
    N = [0] + C + [retorno]

    # Todos los clientes tienen ventana en nuestro formato actual.
    N_TW = C.copy()

    # Depósito de salida y retorno: misma ubicación física.
    coordenadas = {
        0: (deposito["x"], deposito["y"]),
        retorno: (deposito["x"], deposito["y"]),
    }
    coordenadas.update(
        {
            i: (clientes[i]["x"], clientes[i]["y"]) for i in C
        }
    )

    td = {
        (i, j): math.hypot(
            coordenadas[i][0] - coordenadas[j][0],
            coordenadas[i][1] - coordenadas[j][1],
        )
        for i in N
        for j in N
    }

    tt = td.copy()

    q = {
        (i, t): demandas[i][t - 1]
        for i in C
        for t in D
    }

    O = [(i, t) for i in C for t in D if q[i, t] > 0]

    st = {
        (i, t): clientes[i]["servicio"]
        for i, t in O
    }

    # Arcos diarios: solo clientes con pedido ese día.
    A = []

    for t in D:
        activos = [i for i in C if q[i, t] > 0]

        A.append((0, retorno, t))

        A.extend((0, i, t) for i in activos)
        A.extend((i, retorno, t) for i in activos)

        A.extend(
            (i, j, t)
                for i in activos
                for j in activos
                if i != j
        )

    return {
        "nombre": cabecera.get("NAME", path.stem),
        "C": C,
        "D": D,
        "R": R,
        "N": N,
        "N_TW": N_TW,
        "O": O,
        "A": A,
        "retorno": retorno,
        "coordenadas": coordenadas,
        "td": td,
        "tt": tt,
        "q": q,
        "st": st,
        "a": {i: clientes[i]["inicio"] for i in C},
        "b": {i: clientes[i]["fin"] for i in C},
        "Q": capacidad,
        "aW": deposito["apertura"],
        "bW": deposito["cierre"],
    }

In [35]:
# Leer desde la carpeta de instancias generadas.
instancia_1 = leer_multidia(PATH_GENERADAS / "I1.txt")
instancia_2 = leer_multidia(PATH_GENERADAS / "I2.txt")

instancias = {
    "I1": instancia_1,
    "I2": instancia_2,
}

for identificador, datos in instancias.items():
    print(
        f"{identificador}: "
        f"{len(datos['C'])} clientes, "
        f"{len(datos['D'])} días, "
        f"{len(datos['R'])} vehículos, "
        f"capacidad {datos['Q']}"
    )

I1: 20 clientes, 3 días, 3 vehículos, capacidad 98.0
I2: 30 clientes, 3 días, 3 vehículos, capacidad 154.0


# Sección 4 Modelo PLEM (versión reducida del conVRP-SLA)

Versión reducida del modelo de Campelo et al. (2019) para **ruteo consistente multidía**: cada cliente queda asignado a una única ruta (vehículo/conductor) durante todo el horizonte, pero el recorrido de cada ruta puede cambiar de un día a otro. Quedan fuera las fechas de liberación, los múltiples períodos de despacho por cliente y la duración máxima de ruta. La numeración de las restricciones es la del paper; se omiten (6), (13) y (15) por quedar fuera del alcance.

**Conjuntos e índices**

- $C=\{1,\dots,n\}$: clientes. $N=C\cup\{0,n+1\}$: nodos, donde $0$ es la salida y $n+1$ el retorno al depósito.
- $R$: rutas (vehículos). $D$: días.
- $O=\{(i,t): q_{it}>0\}$: pedidos. $N_{TW}\subseteq C$: clientes con ventana horaria (en nuestras instancias, todos).
- $A$: arcos por día. Solo conectan clientes con pedido ese día y los nodos depósito; no hay arcos hacia $0$, desde $n+1$ ni de un nodo a sí mismo. Se agrega $(0,n+1,t)$, con distancia y tiempo cero, para una ruta sin entregas.

**Parámetros:** distancia $td_{ij}$, tiempo de viaje $tt_{ij}$, tiempo de servicio $st_{it}>0$, ventana $[a_i,b_i]$, demanda $q_{it}$, capacidad $Q$, horario del depósito $[a_W,b_W]$.

**Variables de decisión**

| Variable (código)          | Dominio                                         | Significado                                                                   |
|----------------------------|-------------------------------------------------|-------------------------------------------------------------------------------|
| $X^r_{ijt}$ (`x[i,j,t,r]`) | binaria                                         | la ruta $r$ recorre el arco $(i,j)$ el día $t$                                |
| $Z^r_i$ (`z[i,r]`)         | binaria                                         | el cliente $i$ queda asignado a la ruta $r$ (sin índice de día: consistencia) |
| $W_{it}$ (`s[i,t]`)        | $[a_i,b_i]$ (o $[a_W,b_W]$ si no tiene ventana) | inicio del servicio a $i$ el día $t$                                          |
| $dW_r$ (`dW[r]`)           | $[a_W,b_W]$                                     | hora de salida de la ruta $r$ (la misma todos los días)                       |
| $rW^r_t$ (`rW[t,r]`)       | $[a_W,b_W]$                                     | hora de retorno de la ruta $r$ el día $t$                                     |

**Modelo**

$$\min \sum_{(i,j,t)\in A}\sum_{r\in R} td_{ij}\,X^r_{ijt} \tag{1}$$

- (2) Asignación única: $\sum_{r} Z^r_i = 1,\ \forall i\in C$.
- (3) Entrada: $\sum_{i:(i,j,t)\in A} X^r_{ijt} = Z^r_j,\ \forall (j,t)\in O,\ r\in R$.
- (4) Salida: $\sum_{j:(i,j,t)\in A} X^r_{ijt} = Z^r_i,\ \forall (i,t)\in O,\ r\in R$.
- (5) Salida del depósito: $\sum_{j:(0,j,t)\in A} X^r_{0jt} = 1,\ \forall r\in R,\ t\in D$.
- (7) Secuencia entre clientes: $W_{it}+st_{it}+tt_{ij}\le W_{jt}+M^t_{ij}(1-X^r_{ijt})$, para $(i,j,t)\in A$ con $i,j\in C$.
- (8) Ventanas: $a_i\le W_{it}\le b_i,\ \forall (i,t)\in O,\ i\in N_{TW}$.
- (9) Salida y primer cliente: $dW_r+tt_{0j}\le W_{jt}+M^t_{0j}(1-X^r_{0jt})$.
- (10) Último cliente y retorno: $W_{it}+st_{it}+tt_{i,n+1}\le rW^r_t+M^t_{i}(1-X^r_{i,n+1,t})$.
- (11) $dW_r\ge a_W$. (12) $rW^r_t\le b_W$.
- (14) Capacidad: $\sum_{i\in C} q_{it}Z^r_i\le Q,\ \forall r\in R,\ t\in D$.
- (16) Dominios: $X^r_{ijt}, Z^r_i\in\{0,1\}$; $W_{it}, dW_r, rW^r_t\ge 0$.

**Big-M por arco.** Sea $l_i=a_i$ y $u_i=b_i$ si $i\in N_{TW}$ (en otro caso $a_W$ y $b_W$):

$$M^t_{ij}=u_i+st_{it}+tt_{ij}-l_j,\qquad M^t_{0j}=b_W+tt_{0j}-l_j,\qquad M^t_{i}=u_i+st_{it}+tt_{i,n+1}-a_W.$$

Son válidos porque $l_i\le W_{it}\le u_i$, $a_W\le dW_r\le b_W$ y $a_W\le rW^r_t\le b_W$. Con $X=0$ cada restricción queda satisfecha para cualquier valor de las variables, así que no se corta ninguna solución. Un $M$ ajustado por arco también debilita menos la relajación continua que un $M$ global.

**Notas de modelación**

- Las cotas $dW_r\le b_W$ y $rW^r_t\ge a_W$ no están en el paper. No cortan soluciones (salir más temprano nunca perjudica y una ruta vacía no tiene restricciones de tiempo) y son necesarias para los $M$ de (9) y (10).
- (3) y (4) se escriben para cada $(i,t)\in O$ en vez de sumar sobre los días, como en el paper. Con la suma, un cliente con pedidos varios días solo podría recibir una visita en todo el horizonte.
- (7) elimina subrutas porque $st_{it}>0$: un ciclo de clientes desconectado del depósito exigiría $W$ estrictamente creciente sobre sí mismo.
- $Q$ aparece solo como lado derecho de (14). Esto es lo que permite el análisis de sensibilidad de la Sección 8.

**MIP start.** Se usa la solución factible de referencia de la Sección 2 (`*_verificacion.txt`): rutas y arcos de la referencia, $dW_r=a_W$, $W_{it}$ por propagación de tiempos y $rW^r_t$ igual a la hora real de retorno. Gurobi parte con un incumbente y busca mejorarlo.

In [50]:
# ============================================================
# Seccion 4.1: construccion del modelo MIP con Gurobi
# ============================================================
import re

import gurobipy as gp
from gurobipy import GRB


def construir_modelo(instancia, verbo=False):
    """Modelo PLEM reducido. Devuelve (modelo, variables)."""
    C, D, R = instancia["C"], instancia["D"], instancia["R"]
    ret, A, O = instancia["retorno"], instancia["A"], set(instancia["O"])
    NTW = set(instancia["N_TW"])
    q, st, td, tt = (instancia[k] for k in ("q", "st", "td", "tt"))
    a, b, aW, bW, Q = (instancia[k] for k in ("a", "b", "aW", "bW", "Q"))

    # Cotas de W_it: lo[i] <= W_it <= up[i]. Se usan en los Big-M por arco.
    lo = {i: a[i] if i in NTW else aW for i in C}
    up = {i: b[i] if i in NTW else bW for i in C}

    m = gp.Model(f"ConVRP_{instancia['nombre']}")
    m.Params.OutputFlag = 1 if verbo else 0

    # --- Variables ---
    x = {(i, j, t, r): m.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_d{t}_r{r}")
        for (i, j, t) in A for r in R}  # (16)
    z = {(i, r): m.addVar(vtype=GRB.BINARY, name=f"z_{i}_r{r}")
        for i in C for r in R}  # (16)
    s = {(i, t): m.addVar(lb=lo[i], ub=up[i], name=f"s_{i}_d{t}")
        for (i, t) in O}  # W_it: (16) y ventanas (8)
    # dW <= bW y rW >= aW no cortan soluciones y permiten Big-M ajustados.
    dW = {r: m.addVar(lb=aW, ub=bW, name=f"dW_r{r}") for r in R}  # (16) y (11)
    rW = {(t, r): m.addVar(lb=aW, ub=bW, name=f"rW_d{t}_r{r}")
        for t in D for r in R}  # (16) y (12)

    # --- (1) Objetivo: distancia total ---
    m.setObjective(
        gp.quicksum(td[i, j] * x[i, j, t, r] for (i, j, t) in A for r in R),
        GRB.MINIMIZE
    )

    entra, sale = {}, {}
    for (i, j, t) in A:
        sale.setdefault((i, t), []).append(j)
        entra.setdefault((j, t), []).append(i)

    # --- (2) Cada cliente a una unica ruta ---
    for i in C:
        m.addConstr(gp.quicksum(z[i, r] for r in R) == 1, name=f"asig_{i}")

    # --- (3) y (4) Entrada y salida de cada cliente con pedido ---
    for (i, t) in O:
        for r in R:
            m.addConstr(
                gp.quicksum(x[k, i, t, r] for k in entra[i, t]) == z[i, r],
                name=f"entrada_{i}_d{t}_r{r}"
            )
            m.addConstr(
                gp.quicksum(x[i, k, t, r] for k in sale[i, t]) == z[i, r],
                name=f"salida_{i}_d{t}_r{r}"
            )

    # --- (5) Una salida del deposito por ruta y dia ---
    for t in D:
        for r in R:
            m.addConstr(
                gp.quicksum(x[0, j, t, r] for j in sale[0, t]) == 1,
                name=f"deposito_d{t}_r{r}"
            )

    # --- (7), (9), (10) Coordinacion temporal con Big-M por arco ---
    for (i, j, t) in A:
        if i == 0 and j == ret:
            continue  # ruta sin entregas
        if i == 0:  # (9) salida -> primer cliente
            M = bW + tt[0, j] - lo[j]
            for r in R:
                m.addConstr(
                    dW[r] + tt[0, j] <= s[j, t] + M * (1 - x[0, j, t, r]),
                    name=f"sal_dep_{j}_d{t}_r{r}"
                )
        elif j == ret:  # (10) ultimo cliente -> retorno
            M = up[i] + st[i, t] + tt[i, ret] - aW
            for r in R:
                m.addConstr(
                    s[i, t] + st[i, t] + tt[i, ret]
                    <= rW[t, r] + M * (1 - x[i, ret, t, r]),
                    name=f"ret_dep_{i}_d{t}_r{r}"
                )
        else:  # (7) cliente -> cliente
            M = up[i] + st[i, t] + tt[i, j] - lo[j]
            for r in R:
                m.addConstr(
                    s[i, t] + st[i, t] + tt[i, j]
                    <= s[j, t] + M * (1 - x[i, j, t, r]),
                    name=f"sec_{i}_{j}_d{t}_r{r}"
                )

    # --- (14) Capacidad por ruta y dia (Q solo aparece como RHS) ---
    for t in D:
        for r in R:
            m.addConstr(
                gp.quicksum(q[i, t] * z[i, r] for i in C if q[i, t] > 0) <= Q,
                name=f"cap_d{t}_r{r}"
            )

    return m, {"x": x, "z": z, "s": s, "dW": dW, "rW": rW}

In [37]:
# ============================================================
# Seccion 4.2: MIP start desde la solucion de referencia (verificacion)
# ============================================================
def solucion_de_verificacion(path):
    """Extrae rutas y inicios de servicio del archivo *_verificacion.txt."""
    lineas = Path(path).read_text(encoding="utf-8").splitlines()
    rutas, inicios = {}, {}
    patron_ruta = re.compile(r"^Ruta (\d+), dia (\d+): (.+)$")
    patron_inicio = re.compile(r"^Inicio de servicio .*\): (.*)$")
    for linea in lineas:
        linea = linea.strip()
        mr = patron_ruta.match(linea)
        if mr:
            r, t = int(mr.group(1)), int(mr.group(2))
            nodos = [int(x) for x in mr.group(3).split("->")]
            rutas[r, t] = [i for i in nodos[1:-1]]  # sin el depot
            continue
        mi = patron_inicio.match(linea)
        if mi and rutas:
            r, t = list(rutas)[-1]  # ultima ruta leida
            for par in mi.group(1).split(","):
                i, val = par.split(":")
                inicios[int(i), t] = float(val)
    return rutas, inicios


def aplicar_inicio(vars_, instancia, rutas):
    """MIP start desde la solucion de referencia."""
    C, D, R = instancia["C"], instancia["D"], instancia["R"]
    ret = instancia["retorno"]
    st, tt, a, aW = instancia["st"], instancia["tt"], instancia["a"], instancia["aW"]
    x, z, s, dW, rW = (vars_[k] for k in ("x", "z", "s", "dW", "rW"))

    pertenece = {}
    for (r, t), clientes in rutas.items():
        for i in clientes:
            pertenece.setdefault(i, r)

    for clave in x:
        x[clave].Start = 0
    for r in R:
        dW[r].Start = aW
    for t in D:
        for r in R:
            ruta = rutas.get((r, t), [])
            for p, j in zip([0] + ruta, ruta + [ret]):
                x[p, j, t, r].Start = 1
            reloj, anterior = aW, 0
            for i in ruta:
                inicio = max(a[i], reloj + tt[anterior, i])
                s[i, t].Start = inicio
                reloj, anterior = inicio + st[i, t], i
            rW[t, r].Start = reloj + tt[anterior, ret]
    for i in C:
        for r in R:
            z[i, r].Start = 1 if pertenece[i] == r else 0


In [38]:
# ============================================================
# Seccion 4.3: validacion independiente de la solucion del solver
# ============================================================
def validar_solucion(instancia, vars_):
    """Revisa cobertura, capacidad, ventanas, tiempos, retorno y consistencia.
    Devuelve (rutas, distancia_total). Lanza AssertionError si algo falla."""
    C, D, R = instancia["C"], instancia["D"], instancia["R"]
    ret, A, O = instancia["retorno"], instancia["A"], set(instancia["O"])
    q, st, tt = instancia["q"], instancia["st"], instancia["tt"]
    a, b, aW, bW, Q = (instancia[k] for k in ("a", "b", "aW", "bW", "Q"))
    x, s = vars_["x"], vars_["s"]
    dW, rW = vars_["dW"], vars_["rW"]

    vehiculo_de, rutas = {}, {}
    for t in D:
        act = {i for i in C if (i, t) in O}
        atendidos = set()
        for r in R:
            arcos = [(i, j) for (i, j, dd) in A if dd == t and x[i, j, dd, r].X > 0.5]
            succ = dict(arcos)
            if not succ:
                continue
            ruta, nodo = [], 0
            while succ.get(nodo, ret) != ret:
                nodo = succ[nodo]
                assert nodo in act and nodo not in ruta, "subruta o nodo repetido"
                ruta.append(nodo)
            rutas[r, t] = ruta
            carga = sum(q[i, t] for i in ruta)
            assert carga <= Q, f"capacidad dia {t} vehiculo {r}: {carga} > {Q}"
            reloj, anterior = dW[r].X, 0
            for i in ruta:
                inicio = s[i, t].X
                assert a[i] - 1e-6 <= inicio <= b[i] + 1e-6, \
                    f"ventana cliente {i} dia {t}"
                assert inicio >= reloj + tt[anterior, i] - 1e-6, \
                    f"tiempo de viaje violado en {i} dia {t}"
                reloj = inicio + st[i, t]
                anterior = i
                atendidos.add(i)
                # consistencia: mismo vehiculo en todos los dias
                if i in vehiculo_de:
                    assert vehiculo_de[i] == r, f"vehiculo inconsistente en {i}"
                else:
                    vehiculo_de[i] = r
            retorno = reloj + tt[anterior, ret]
            assert retorno <= rW[t, r].X + 1e-6, f"rW menor al retorno real dia {t} vehiculo {r}"
            assert retorno <= bW + 1e-6, f"retorno tardio dia {t} vehiculo {r}: {retorno}"
        assert atendidos == act, f"cobertura incorrecta dia {t}: {atendidos ^ act}"

    dist = sum(
        instancia["td"][i, j] * x[i, j, t, r].X
            for (i, j, t) in instancia["A"] for r in R
    )
    return rutas, dist


# Sección 5 Resolución y comparación de configuraciones

Todas las corridas usan el mismo protocolo: `Threads=1`, `TimeLimit=600`, el mismo MIP start y los demás parámetros de Gurobi en su valor por defecto. En cada configuración se cambia un único parámetro respecto del baseline:

| Configuración           | Parámetro                   |
|-------------------------|-----------------------------|
| Baseline                | por defecto                 |
| Cuts=0                  | `Cuts=0`                    |
| Presolve=0              | `Presolve=0`                |
| MIPFocus=1 / MIPFocus=3 | `MIPFocus=1` / `MIPFocus=3` |
| Heuristics=0            | `Heuristics=0`              |

Cada solución se valida con `validar_solucion` (flujo, consistencia de ruta, capacidad, ventanas, horarios de salida y retorno). El resultado queda en la columna `valida`: una falla se registra sin abortar el resto de las corridas.

La tabla reporta objetivo, cota inferior, `gap_%`, tiempo, nodos explorados y estado (`OPTIMO` o `TIME_LIMIT`).

In [51]:
# ============================================================
# Seccion 4.4: resolver una instancia y reportar
# ============================================================
import pandas as pd

ESTADOS = {
    GRB.OPTIMAL: "OPTIMO", GRB.TIME_LIMIT: "TIME_LIMIT",
    GRB.INFEASIBLE: "INFACTIBLE", GRB.INTERRUPTED: "INTERRUMPIDO"
}


def cargar_start(nombre, vars_):
    archivo = PATH_GENERADAS / f"{nombre}_verificacion.txt"
    if archivo.is_file():
        rutas_ref, _ = solucion_de_verificacion(archivo)
        aplicar_inicio(vars_, instancias[nombre], rutas_ref)


def correr(nombre, params=None, tiempo_limite=600, usar_start=True, prm=None, verbo=False):
    """Una corrida del MILP con Threads=1 y TimeLimit fijos.
    `params` es el unico cambio a probar; `prm` es un archivo de parametros de tune()."""
    inst = instancias[nombre]
    m, vars_ = construir_modelo(inst, verbo=verbo)
    if prm:
        m.read(str(prm))
        m.Params.OutputFlag = 1 if verbo else 0
    m.Params.Threads = 1
    m.Params.TimeLimit = tiempo_limite
    for k, val in (params or {}).items():
        m.setParam(k, val)
    if usar_start:
        cargar_start(nombre, vars_)

    m.optimize()
    hay = m.SolCount > 0
    valida, rutas = "-", None
    if hay:
        try:
            rutas, _ = validar_solucion(inst, vars_)
            valida = "OK"
        except AssertionError as e:
            valida = f"FALLA: {e}"
    return {
        "modelo": m, "vars": vars_, "rutas": rutas, "valida": valida,
        "estado": ESTADOS.get(m.Status, m.Status),
        "objetivo": m.ObjVal if hay else None,
        "cota": m.ObjBound if hay else None,
        "gap": 100 * m.MIPGap if hay else None,
        "tiempo": m.Runtime, "nodos": m.NodeCount
    }


def fila(etiqueta, res):
    return {
        "config": etiqueta, "objetivo": res["objetivo"], "cota": res["cota"],
        "gap_%": res["gap"], "tiempo_s": res["tiempo"], "nodos": res["nodos"],
        "estado": res["estado"], "valida": res["valida"]
    }


def mejor(nombre, claves):
    """(config, corrida) con menor objetivo; a igualdad, menor gap y luego menor tiempo."""
    cand = [(c, resultados[nombre, c]) for c in claves
        if resultados[nombre, c]["objetivo"] is not None]
    return min(
        cand, key=lambda p: (
            round(p[1]["objetivo"], 2),
            round(p[1]["gap"], 1), p[1]["tiempo"]
        )
    )


def relajacion(nombre):
    m, _ = construir_modelo(instancias[nombre])
    m.update()  # procesar el modelo antes de relajar
    m.Params.Threads = 1
    lp = m.relax()  # x, z en [0,1]
    assert lp.NumVars == m.NumVars, "relajacion sin variables"
    lp.optimize()
    assert lp.Status == GRB.OPTIMAL, "relajacion no optima"
    return lp.ObjVal


def sintonizar(nombre, tune_limite=3600, limite_corrida=60, usar_start=True):
    """tune() con corridas internas cortas: con TimeLimit=600 y 3 semillas un solo
    candidato costaba 1800 s y tune() no alcanzaba a probar nada mas que el baseline."""
    m, vars_ = construir_modelo(instancias[nombre], verbo=True)
    m.Params.Threads = 1
    m.Params.TimeLimit = limite_corrida  # limite de cada corrida interna de tune()
    m.Params.TuneTrials = 2              # semillas por candidato
    m.Params.TuneTimeLimit = tune_limite
    if usar_start:
        cargar_start(nombre, vars_)
    m.tune()
    if m.TuneResultCount == 0:
        print(f"[{nombre}] tune() no encontro parametros mejores que los por defecto")
        return None
    m.getTuneResult(0)
    prm = Path("results") / f"{nombre}_tune.prm"
    prm.parent.mkdir(parents=True, exist_ok=True)
    m.write(str(prm))
    cambios = [l for l in prm.read_text().splitlines()
        if l and not l.startswith(("#", "Threads", "TimeLimit", "Tune", "OutputFlag", "LicenseID"))]
    print(f"[{nombre}] parametros de tune():", cambios)
    return prm


def sensibilidad(nombre):
    """Fija las enteras de la mejor corrida y analiza el RHS (Q) de las restricciones (14)."""
    inst = instancias[nombre]
    f = mejor(nombre, [c for (n, c) in resultados if n == nombre])[1]["modelo"].fixed()
    f.Params.OutputFlag = 0
    f.Params.Presolve = 0  # sin presolve y con simplex: hay base para SARHS
    f.Params.Method = 1
    f.optimize()
    assert f.Status == GRB.OPTIMAL, "modelo fijado no optimo"
    filas = []
    for t in inst["D"]:
        for r in inst["R"]:
            c = f.getConstrByName(f"cap_d{t}_r{r}")
            filas.append(
                {
                    "dia": t, "veh": r, "carga": inst["Q"] - c.Slack, "Q": inst["Q"],
                    "holgura": c.Slack, "precio_sombra": c.Pi,
                    "SARHSLow": c.SARHSLow, "SARHSUp": c.SARHSUp
                }
            )
    df = pd.DataFrame(filas)
    for col in ("SARHSLow", "SARHSUp"):
        df[col] = df[col].apply(lambda v: math.inf if v >= 1e20 else v)
    return df

In [40]:
# ============================================================
# Seccion 4.5: resolver ambas instancias
# ============================================================
CONFIGS = {
    "Baseline": {},
    "Cuts=0": {"Cuts": 0},
    "Presolve=0": {"Presolve": 0},
    "MIPFocus=1": {"MIPFocus": 1},
    "MIPFocus=3": {"MIPFocus": 3},
    "Heuristics=0": {"Heuristics": 0},
}
# El MIP start enmascara el efecto de las heuristicas: se repiten estas dos sin start.
SIN_START = [f"{c} (sin start)" for c in ("Baseline", "Heuristics=0")]

resultados = {}
for nombre in instancias:
    for cfg, params in CONFIGS.items():
        resultados[nombre, cfg] = correr(nombre, params)
        print(nombre, fila(cfg, resultados[nombre, cfg]))
    print(f"\n=== {nombre} ===")
    print(pd.DataFrame([fila(c, resultados[nombre, c]) for c in CONFIGS]).round(2).to_string(index=False))

    for cfg in ("Baseline", "Heuristics=0"):
        resultados[nombre, f"{cfg} (sin start)"] = correr(nombre, CONFIGS[cfg], usar_start=False)
    print(f"\n=== {nombre} (sin MIP start) ===")
    print(pd.DataFrame([fila(c, resultados[nombre, c]) for c in SIN_START]).round(2).to_string(index=False))


=== I1 ===
      config  objetivo   cota  gap_%  tiempo_s    nodos     estado valida
    Baseline    876.55 813.53   7.19    600.00 296847.0 TIME_LIMIT     OK
      Cuts=0    882.82 711.05  19.46    600.00 502094.0 TIME_LIMIT     OK
  Presolve=0    876.55 781.81  10.81    600.00 278898.0 TIME_LIMIT     OK
  MIPFocus=1    870.55 768.71  11.70    600.00 502601.0 TIME_LIMIT     OK
  MIPFocus=3    870.55 815.18   6.36    600.01  20658.0 TIME_LIMIT     OK
Heuristics=0    883.15 821.10   7.03    600.00 353885.0 TIME_LIMIT     OK

=== I2 ===
      config  objetivo    cota  gap_%  tiempo_s    nodos     estado valida
    Baseline   1440.88 1040.73  27.77    600.00  43876.0 TIME_LIMIT     OK
      Cuts=0   1440.88  931.87  35.33    600.00 107833.0 TIME_LIMIT     OK
  Presolve=0   1352.91 1010.45  25.31    600.00  40041.0 TIME_LIMIT     OK
  MIPFocus=1   1440.88 1013.28  29.68    600.00  79234.0 TIME_LIMIT     OK
  MIPFocus=3   1440.88 1076.55  25.29    600.01   1609.0 TIME_LIMIT     OK
Heuristi

# Sección 6 Relajación continua

Se relajan $X^r_{ijt}, Z^r_i\in\{0,1\}$ a $[0,1]$ (`Model.relax()`) y se resuelve el LP. Se reporta $z_{LP}$ y la brecha de integralidad respecto del mejor incumbente $z_{MILP}$ entre las seis configuraciones de la Sección 5:

$$\text{brecha}=\frac{z_{MILP}-z_{LP}}{z_{MILP}}\times 100\%$$

Interpretación: la relajación amplía la región factible y mantiene el mismo objetivo, por lo que $z_{LP}\le z_{MILP}$ (dualidad débil) y entrega una cota inferior válida. Qué tan ajustada es depende de la formulación: la relajación admite $X$ y $Z$ fraccionarios, y con ellos las restricciones Big-M (7), (9) y (10) se debilitan.

In [48]:
filas = []
for nombre in instancias:
    z_milp = mejor(nombre, list(CONFIGS))[1]["objetivo"]
    z_lp = relajacion(nombre)
    filas.append(
        {
            "instancia": nombre, "z_LP": z_lp, "z_MILP": z_milp,
            "brecha_%": 100 * (z_milp - z_lp) / z_milp
        }
    )
print(pd.DataFrame(filas).round(2).to_string(index=False))

instancia   z_LP  z_MILP  brecha_%
       I1 585.05  870.55     32.80
       I2 798.10 1352.91     41.01


# Sección 7 Ajuste automático de parámetros (`Model.tune()`)

Se ejecuta `Model.tune()` por instancia con `TuneTimeLimit=3600` s, `TuneTrials=2`, `Threads=1` y `TimeLimit=60` s como límite de cada corrida interna. Con `TimeLimit=600` y 3 semillas, un solo candidato costaba 1800 s y `tune()` no probaba nada más que el baseline. Los parámetros del mejor resultado se guardan en `results/<instancia>_tune.prm` y se vuelve a resolver con el protocolo de la Sección 5 (`Threads=1`, `TimeLimit=600`). Se compara contra el baseline y contra la mejor configuración manual (la mejor entre las cinco variantes, sin contar el baseline). Si `tune()` no encuentra parámetros mejores que los por defecto, no se genera el `.prm` y la corrida `tune` equivale al baseline.

In [42]:
for nombre in instancias:
    prm = sintonizar(nombre)
    resultados[nombre, "tune"] = correr(nombre, prm=prm)
    cfg_m, res_m = mejor(nombre, [c for c in CONFIGS if c != "Baseline"])
    print(f"\n=== {nombre} ===")
    print(
        pd.DataFrame(
            [
                fila("Baseline", resultados[nombre, "Baseline"]),
                fila(f"Mejor manual ({cfg_m})", res_m),
                fila("tune()", resultados[nombre, "tune"])
            ]
        ).round(2).to_string(index=False)
    )

Set parameter OutputFlag to value 1
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
Set parameter TuneTimeLimit to value 1800
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Linux Mint 22.3")

CPU model: AMD Ryzen 5 5600X 6-Core Processor, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
Threads  1
TuneTimeLimit  1800

Start tuning.
Tune 34 parameters.

Solving model using baseline parameter set with TimeLimit=600s

Total tuning time limit is 1800s
-------------------------------------------------------------------------------

Testing candidate parameter set 1...

	Threads 1 (fixed)

Solving ConVRP_I1 (Min) with random seed #1 ... MIP gap 7.29%
Solving ConVRP_I1 (Min) with random seed #2 ... MIP gap 6.07%
Solving ConVRP_I1 (Min) with random seed #3 ... MIP gap 5.23%

Summary candidate parameter set 1 
 # Name              0        1        2      Avg  

# Sección 8 Análisis de sensibilidad de la capacidad $Q$

Sobre la mejor corrida de cada instancia (menor objetivo; a igualdad, menor gap y menor tiempo) se fijan las variables enteras con `Model.fixed()` y se resuelve el LP resultante, con `Presolve=0` y simplex para disponer de una base. Para cada restricción de capacidad (14), una por (día, ruta), se reportan la carga, la holgura, el precio sombra (`Pi`) y el rango de validez del lado derecho (`SARHSLow`, `SARHSUp`).

Lectura de los resultados:

- Al fijar $X$ y $Z$ el objetivo queda constante, por lo que se espera un precio sombra nulo. Es una propiedad de la fijación: con la asignación fija, más capacidad no reduce la distancia. Para aprovecharla hay que resolver de nuevo el MILP, porque la asignación puede cambiar.
- `SARHSLow` es la carga de la ruta en ese día, y `SARHSUp` es infinito. El valor $\max \text{SARHSLow}$ es el $Q$ mínimo con el que la solución fija sigue siendo factible.

In [43]:
for nombre in instancias:
    df = sensibilidad(nombre)
    print(f"\n=== {nombre} ===")
    print(df.round(3).to_string(index=False))
    print(
        "Q minimo que mantiene factible la asignacion fija:", df["SARHSLow"].max(),
        "| Q actual:", instancias[nombre]["Q"]
    )


=== I1 ===
 dia  veh  carga    Q  holgura  precio_sombra  SARHSLow  SARHSUp
   1    1   92.0 98.0      6.0            0.0      92.0      inf
   1    2   42.0 98.0     56.0            0.0      42.0      inf
   1    3   96.0 98.0      2.0            0.0      96.0      inf
   2    1   71.0 98.0     27.0            0.0      71.0      inf
   2    2   58.0 98.0     40.0            0.0      58.0      inf
   2    3   92.0 98.0      6.0            0.0      92.0      inf
   3    1   70.0 98.0     28.0            0.0      70.0      inf
   3    2   41.0 98.0     57.0            0.0      41.0      inf
   3    3   63.0 98.0     35.0            0.0      63.0      inf
Q minimo que mantiene factible la asignacion fija: 96.0 | Q actual: 98.0

=== I2 ===
 dia  veh  carga     Q  holgura  precio_sombra  SARHSLow  SARHSUp
   1    1  105.0 154.0     49.0            0.0     105.0      inf
   1    2  125.0 154.0     29.0            0.0     125.0      inf
   1    3  128.0 154.0     26.0            0.0     128.

# Sección 9 Exportación de resultados

Se guarda en `results/` para cada instancia: la tabla de configuraciones (`*_configuraciones.csv`), las rutas de cada corrida (`*_solver.txt`), la relajación y su brecha (`*_relajacion.csv`) y la sensibilidad de $Q$ (`*_sensibilidad.csv`).

In [44]:
RES = Path("results")
RES.mkdir(parents=True, exist_ok=True)

for nombre in instancias:
    claves = [c for (n, c) in resultados if n == nombre]

    # Tabla de configuraciones + corridas sin start + tune
    pd.DataFrame([fila(c, resultados[nombre, c]) for c in claves]) \
        .round(4).to_csv(RES / f"{nombre}_configuraciones.csv", index=False)

    # Rutas de cada corrida
    with open(RES / f"{nombre}_solver.txt", "w", encoding="utf-8") as f:
        for c in claves:
            res = resultados[nombre, c]
            f.write(f"=== {nombre} | {c} | {res['estado']} | obj={res['objetivo']} | gap={res['gap']}% ===\n")
            for k, v in sorted((res["rutas"] or {}).items()):
                f.write(f"{k}: {v}\n")
            f.write("\n")

    # Relajacion y sensibilidad (son LP, rapidas)
    z_milp = mejor(nombre, [c for c in claves if c in CONFIGS])[1]["objetivo"]
    z_lp = relajacion(nombre)
    pd.DataFrame(
        [
            {
                "instancia": nombre, "z_LP": z_lp, "z_MILP": z_milp,
                "brecha_%": 100 * (z_milp - z_lp) / z_milp
            }
        ]
    ) \
        .round(4).to_csv(RES / f"{nombre}_relajacion.csv", index=False)
    sensibilidad(nombre).round(4).to_csv(RES / f"{nombre}_sensibilidad.csv", index=False)